In [0]:
SELECT *
FROM workspace.default.wa_fn_use_c_telco_customer_churn
LIMIT 10

customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes
9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.1,1949.4,No
6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.8,3046.05,Yes
6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,Yes,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


In [0]:
SELECT 
    COUNT(*)                                          AS total_clientes,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)   AS total_churn,
    ROUND(
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
    )                                                 AS taxa_churn_pct,
    SUM(CASE WHEN TotalCharges = ' ' THEN 1 ELSE 0 END) AS total_charges_vazios
FROM workspace.default.wa_fn_use_c_telco_customer_churn

total_clientes,total_churn,taxa_churn_pct,total_charges_vazios
7043,1869,26.54,11


In [0]:
SELECT 
    Contract,
    COUNT(*)                                                AS total_clientes,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)         AS total_churn,
    ROUND(
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
    )                                                       AS taxa_churn_pct
FROM workspace.default.wa_fn_use_c_telco_customer_churn
GROUP BY Contract
ORDER BY taxa_churn_pct DESC

Contract,total_clientes,total_churn,taxa_churn_pct
Month-to-month,3875,1655,42.71
One year,1473,166,11.27
Two year,1695,48,2.83


In [0]:
SELECT 
    PaymentMethod,
    COUNT(*)                                                AS total_clientes,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)         AS total_churn,
    ROUND(
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
    )                                                       AS taxa_churn_pct
FROM workspace.default.wa_fn_use_c_telco_customer_churn
GROUP BY PaymentMethod
ORDER BY taxa_churn_pct DESC

PaymentMethod,total_clientes,total_churn,taxa_churn_pct
Electronic check,2365,1071,45.29
Mailed check,1612,308,19.11
Bank transfer (automatic),1544,258,16.71
Credit card (automatic),1522,232,15.24


In [0]:
WITH clientes_com_faixa AS (
    SELECT 
        Churn,
        CASE 
            WHEN tenure BETWEEN 0  AND 12  THEN '01 - 0 a 12 meses'
            WHEN tenure BETWEEN 13 AND 24  THEN '02 - 13 a 24 meses'
            WHEN tenure BETWEEN 25 AND 48  THEN '03 - 25 a 48 meses'
            WHEN tenure > 48               THEN '04 - Mais de 48 meses'
        END AS faixa_tenure
    FROM workspace.default.wa_fn_use_c_telco_customer_churn
)
SELECT 
    faixa_tenure,
    COUNT(*)                                                AS total_clientes,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)         AS total_churn,
    ROUND(
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
    )                                                       AS taxa_churn_pct
FROM clientes_com_faixa
GROUP BY faixa_tenure
ORDER BY faixa_tenure

faixa_tenure,total_clientes,total_churn,taxa_churn_pct
01 - 0 a 12 meses,2186,1037,47.44
02 - 13 a 24 meses,1024,294,28.71
03 - 25 a 48 meses,1594,325,20.39
04 - Mais de 48 meses,2239,213,9.51


In [0]:

WITH base AS (
    SELECT
        Contract,
        PaymentMethod,
        COUNT(*)                                                    AS total_clientes,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)             AS total_churn,
        ROUND(
            SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
        )                                                           AS taxa_churn_pct
    FROM workspace.default.wa_fn_use_c_telco_customer_churn
    GROUP BY Contract, PaymentMethod
),
ranking AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY Contract 
            ORDER BY taxa_churn_pct DESC
        ) AS rank_risco_no_contrato
    FROM base
)
SELECT *
FROM ranking
ORDER BY Contract, rank_risco_no_contrato

Contract,PaymentMethod,total_clientes,total_churn,taxa_churn_pct,rank_risco_no_contrato
Month-to-month,Electronic check,1850,994,53.73,1
Month-to-month,Bank transfer (automatic),589,201,34.13,2
Month-to-month,Credit card (automatic),543,178,32.78,3
Month-to-month,Mailed check,893,282,31.58,4
One year,Electronic check,347,64,18.44,1
One year,Credit card (automatic),398,41,10.30,2
One year,Bank transfer (automatic),391,38,9.72,3
One year,Mailed check,337,23,6.82,4
Two year,Electronic check,168,13,7.74,1
Two year,Bank transfer (automatic),564,19,3.37,2


In [0]:
WITH base AS (
    SELECT
        InternetService,
        COUNT(*)                                                AS total_clientes,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)         AS total_churn,
        ROUND(
            SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
        )                                                       AS taxa_churn_pct,
        ROUND(AVG(CAST(MonthlyCharges AS DOUBLE)), 2)           AS ticket_medio
    FROM workspace.default.wa_fn_use_c_telco_customer_churn
    GROUP BY InternetService
),
com_media_geral AS (
    SELECT 
        *,
        ROUND(AVG(taxa_churn_pct) OVER (), 2) AS media_geral_churn
    FROM base
)
SELECT 
    InternetService,
    total_clientes,
    total_churn,
    taxa_churn_pct,
    ticket_medio,
    media_geral_churn,
    ROUND(taxa_churn_pct - media_geral_churn, 2) AS desvio_da_media
FROM com_media_geral
ORDER BY taxa_churn_pct DESC

InternetService,total_clientes,total_churn,taxa_churn_pct,ticket_medio,media_geral_churn,desvio_da_media
Fiber optic,3096,1297,41.89,91.5,22.75,19.14
DSL,2421,459,18.96,58.1,22.75,-3.79
No,1526,113,7.40,21.08,22.75,-15.35
